# 🧹 Checkin Data Cleaning & Silver Promotion

> **Layer Transition:** Bronze Final → Silver Cleaned  
> **Source Table:** `yelp_poc_uc.bronze_final.yelp_checkin` (131,930 rows — 1 row per business)  
> **Target Table:** `yelp_poc_uc.silver_cleaned.yelp_checkin` (1 row per unique checkin event, with `checkin_count`)  
> **DQ Reference:** `02_Checkin_DQ_Issues_and_Solutions.md`

This notebook implements the **Bronze → Silver Cleaned data population** for the Yelp Checkin dataset. It applies the chosen resolutions from the Data Quality Issues & Solutions document to produce a clean, normalized, analytics-ready Silver Cleaned table.

> **⚠️ Prerequisites — Run These First (in order):**  
> 1. **`01_Business_Silver_Cleaned`** (DDL query) — creates `yelp_poc_uc.silver_cleaned` schema and the empty `yelp_business` table structure (FK dependency)  
> 2. **`01_Business_Cleaning`** (notebook) — populates `yelp_poc_uc.silver_cleaned.yelp_business` with business IDs so that referential integrity (FK) constraints are satisfied when checkin data is inserted  
> 3. **`02_Checkin_Silver_Cleaned`** (DDL query) — creates the empty `yelp_poc_uc.silver_cleaned.yelp_checkin` table with PK/FK constraints  
>  
> This notebook then **populates** the checkin table created by step 3. The DDL queries are idempotent and can be re-run safely.

## 📍 Pipeline Context

| Step | Notebook | UC Location | Layer |
| ---: | --- | --- | --- |
| 1 | `03_Extract_Yelp_Dataset_from_Kaggle` | `/Volumes/yelp_poc_uc/bronze_landing/landing_zone/yelp_textual_dataset/` | Landing Zone |
| 2 | `01_Landing_to_Staging_Schema_Validator` | `yelp_poc_uc.bronze_staging` | Bronze Staging |
| 3 | `02_Staging_Quality_Gate_and_Promotion` | `yelp_poc_uc.bronze_final.yelp_checkin` | Bronze Final |
| 4 | `4.1.2_Checkin_Basic_Profile` | _(reads bronze_final)_ | Basic EDA |
| 5 | `4.2.2_Checkin_Deep_EDA` | _(reads bronze_final)_ | Deep EDA / DQ Assessment |
| 6a | `01_Business_Silver_Cleaned` (DDL) | `yelp_poc_uc.silver_cleaned.yelp_business` (schema) | Silver Cleaned DDL |
| 6b | `01_Business_Cleaning` | `yelp_poc_uc.silver_cleaned.yelp_business` (data) | Silver Cleaned |
| 6c | `02_Checkin_Silver_Cleaned` (DDL) | `yelp_poc_uc.silver_cleaned.yelp_checkin` (schema) | Silver Cleaned DDL |
| **6d** | **`02_Checkin_Cleaning` ◄ THIS NOTEBOOK** | **`yelp_poc_uc.silver_cleaned.yelp_checkin` (data)** | **Silver Cleaned** |
| 7 | Gold Aggregation _(planned)_ | `yelp_poc_uc.gold` _(planned)_ | Gold |

## ✅ Scope — Core DQ Issues Resolved in This Notebook

This notebook addresses the **structural and integrity** issues that must be fixed to produce a valid Silver table. The output schema is intentionally minimal — **three core columns:**

| # | Column | Type | Description |
| --- | --- | --- | --- |
| 1 | `business_id` | `string` | Unique Yelp business identifier (FK to yelp_business) |
| 2 | `checkin_timestamp` | `timestamp` | Individual checkin event timestamp (ISO 8601) |
| 3 | `checkin_count` | `bigint` | Number of checkins at this exact (business_id, timestamp) pair — aggregated from duplicate rows |

No `_ingestion_timestamp` column — lineage is tracked via TBLPROPERTIES metadata (see DDL rule R6).

### DQ Issues Addressed

| DQ Issue | Summary | Resolution Applied |
| --- | --- | --- |
| **DQ-CHECKIN-001** | `date` column stored as `string` instead of `array<timestamp>` | `split()` → `explode()` → `to_timestamp()` to normalize to one row per event |
| **DQ-CHECKIN-002** | 92.7% of rows contain packed multi-value timestamps (1NF violation) | Same as above — shared root cause |
| **DQ-CHECKIN-005** | 3,543 exact duplicate rows from source data (composite key violations) | `GROUP BY` + `COUNT(*)` → `checkin_count` column — duplicates are aggregated into a counter instead of quarantined |

> **⚠️ Note on DQ-CHECKIN-005:** The RCA confirmed these duplicates originate from the source data, but **cannot definitively determine their intent** — the checkin dataset has no `user_id` column. An alternative interpretation is that these represent **genuine concurrent activity** (multiple users checking in simultaneously, venues with multiple counters/cashiers, or groups arriving together). The DQ-006 RCA already proved that 1–60s-apart checkins are real concurrent behavior; 0-second gaps are a natural extension of that pattern. The `checkin_count` column preserves this signal — a count > 1 at a given (business_id, timestamp) may represent either source duplicates or genuine concurrent activity. No data is discarded; downstream consumers can interpret counts > 1 according to their analytical context. At 0.027% of total rows, the impact is negligible under either interpretation. See the full **Alternative Interpretation — Assumptions & Caveats** section in `02_Checkin_DQ_Issues_and_Solutions.md` for detailed evidence.

## 🔜 Out of Scope — Deferred to Analytics Phase

The DQ document recommends several **additional flag and derived columns** (e.g., `is_complete_year`, `is_near_duplicate`, `is_high_volume_day`, `is_covid_period`, `is_truncated_year`, `is_zero_span`, `activity_tier`, `activity_profile`). These columns are **not added in this notebook** — the Silver Cleaned table stays lean with just the three core columns (`business_id`, `checkin_timestamp`, `checkin_count`).

These enrichments may be considered later during the **analytics or Gold-layer phase**, where the specific analytical use case will determine which flags and features are actually needed. Keeping the Silver table minimal ensures it remains a general-purpose, reusable foundation without premature analytical assumptions baked in.

| Deferred DQ Issue | What It Would Add | Why Deferred |
| --- | --- | --- |
| DQ-CHECKIN-004 | `is_complete_year` flag (2009/2022 partial years) | Analytical filter — context-dependent |
| DQ-CHECKIN-006 | `is_near_duplicate` flag (≤60s gap) | RCA confirmed genuine behavior — flagging is optional |
| DQ-CHECKIN-008 | `is_high_volume_day` flag (>50 checkins/day) | Requires cross-reference with `yelp_business` |
| DQ-CHECKIN-009 | `is_covid_period` + `is_truncated_year` flags | Temporal context — applied at query time |
| DQ-CHECKIN-010 | `activity_tier` + `log1p` transform | Modeling-specific — varies by use case |
| DQ-CHECKIN-011 | `is_zero_span` + `activity_profile` composite | Segmentation — belongs in analytics layer |
| DQ-CHECKIN-007 | LEFT JOIN enforcement for checkin↔tip | Join strategy — applied at query time, not stored |
| DQ-CHECKIN-012 | Temporal caveats for 2020–2022 | Documentation concern — no schema change needed |

In [0]:
# ============================================================
# 🔧 Environment Setup — Prerequisite Check
# ============================================================
# PREREQUISITES (run these BEFORE this notebook, in order):
#   1. 01_Business_Silver_Cleaned (DDL) — creates silver_cleaned
#      schema and empty yelp_business table structure
#   2. 01_Business_Cleaning (notebook) — populates yelp_business
#      with business IDs (required for FK referential integrity)
#   3. 02_Checkin_Silver_Cleaned (DDL) — creates the empty
#      silver_cleaned.yelp_checkin table with PK/FK constraints
#
# This notebook POPULATES the checkin table created by step 3.
# No schema creation is needed here.

# Verify the target table exists (created by DDL query)
try:
    tbl_info = spark.sql("DESCRIBE TABLE yelp_poc_uc.silver_cleaned.yelp_checkin").collect()
    print("✅ Target table exists: yelp_poc_uc.silver_cleaned.yelp_checkin")
    print(f"   Columns: {len([r for r in tbl_info if r['col_name'] and not r['col_name'].startswith('#')])}")
except Exception as e:
    raise RuntimeError(
        "❌ Target table yelp_poc_uc.silver_cleaned.yelp_checkin does not exist!\n"
        "   Run the prerequisites first (in order):\n"
        "   1. 01_Business_Silver_Cleaned (DDL)\n"
        "   2. 01_Business_Cleaning (notebook)\n"
        "   3. 02_Checkin_Silver_Cleaned (DDL)\n"
        f"   Error: {e}"
    )

---

## 🔧 Cleaning Pipeline

The pipeline below applies the three core DQ resolutions in sequence:

1. **Normalize** (DQ-001 / DQ-002) — `split()` → `explode()` → `to_timestamp()` transforms the packed multi-value `date` string into individual checkin event rows
2. **Aggregate** (DQ-005) — `GROUP BY (business_id, checkin_timestamp)` with `COUNT(*)` → `checkin_count` — duplicates are folded into a counter column instead of being quarantined. This preserves the duplicate signal (count > 1) for downstream interpretation.
3. **Populate** — `INSERT INTO` the DDL-created `silver_cleaned.yelp_checkin` table (no `_ingestion_timestamp` — lineage via TBLPROPERTIES per rule R6)

In [0]:
from pyspark.sql.functions import (
    col, split, explode, trim, to_timestamp,
    count, lit, current_timestamp, sum as _sum
)

# ============================================================
# Phase 1: Read Bronze & Normalize (DQ-CHECKIN-001 / DQ-CHECKIN-002)
# ============================================================
# Bronze: 1 row per business, date = comma-separated timestamp string
# After explode: 1 row per checkin event (may contain duplicates)
# Next phase will GROUP BY + COUNT to produce the final silver_cleaned rows

df_bronze = spark.read.table("yelp_poc_uc.bronze_final.yelp_checkin")

df_exploded = (
    df_bronze
    .select(
        col("business_id"),
        explode(split(col("date"), ",")).alias("_raw")
    )
    .withColumn("checkin_timestamp", to_timestamp(trim(col("_raw")), "yyyy-MM-dd HH:mm:ss"))
    .drop("_raw")
)

bronze_count = df_bronze.count()
exploded_count = df_exploded.count()

print(f"📥 Bronze source:    {bronze_count:>12,} rows  (1 per business)")
print(f"💥 After explode:    {exploded_count:>12,} rows  (1 per checkin event)")
print(f"   Expansion factor: {exploded_count / bronze_count:>12.1f}x")

In [0]:
# ============================================================
# Phase 2: Aggregate Duplicates → checkin_count (DQ-CHECKIN-005)
# ============================================================
# Instead of deduplicating and quarantining duplicate rows,
# we GROUP BY the composite key and COUNT(*) to produce a
# checkin_count column. This:
#   • Preserves the duplicate signal (count > 1) for downstream use
#   • Eliminates the need for a separate quarantine table
#   • Produces one row per unique (business_id, checkin_timestamp)
#
# Note: See DQ-005 Alternative Interpretation in the DQ document.
# A checkin_count > 1 MAY represent genuine concurrent activity
# (multiple users, multiple cashiers) rather than source errors.
# The counter approach preserves this information for downstream
# consumers to interpret according to their analytical context.

COMPOSITE_KEY = ["business_id", "checkin_timestamp"]

# Aggregate: one row per unique composite key with occurrence count
df_silver = (
    df_exploded
    .groupBy(COMPOSITE_KEY)
    .agg(count("*").alias("checkin_count"))
)

# Summary stats
total_keys = df_silver.count()
keys_with_dupes = df_silver.filter(col("checkin_count") > 1).count()

print("✅ Phase 2 complete — Silver DataFrame prepared (GROUP BY + COUNT)")
print(f"   Unique composite keys:  {total_keys:>12,}")
print(f"   Keys with count > 1:    {keys_with_dupes:>12,}  (duplicates aggregated into counter)")

In [0]:
# ============================================================
# 📊 DQ-CHECKIN-005 — Aggregation Distribution Report
# ============================================================
# Shows how duplicate rows were folded into checkin_count.
# Most rows have checkin_count=1; a small number have count>1.

from pyspark.sql.functions import max as _max, min as _min, avg as _avg

silver_count = df_silver.count()
agg_stats = df_silver.agg(
    _sum("checkin_count").alias("total_events"),
    _max("checkin_count").alias("max_count"),
    _avg("checkin_count").alias("avg_count"),
    count("*").alias("unique_keys")
).collect()[0]

total_events = int(agg_stats["total_events"])
max_count    = int(agg_stats["max_count"])
avg_count    = float(agg_stats["avg_count"])

# Breakdown by checkin_count value
count_dist = (
    df_silver
    .groupBy("checkin_count")
    .agg(count("*").alias("num_keys"))
    .orderBy("checkin_count")
    .collect()
)

keys_gt_1 = sum(r["num_keys"] for r in count_dist if r["checkin_count"] > 1)
excess    = total_events - silver_count  # rows folded into counters

print("=" * 65)
print("  📊 DQ-CHECKIN-005 — Aggregation Distribution Report")
print("=" * 65)

print(f"\n  ── Overall Pipeline Impact ──")
print(f"  📥 Total rows after explode:          {exploded_count:>12,}")
print(f"  ✅ Unique composite keys (silver):    {silver_count:>12,}")
print(f"  📊 Total events (sum checkin_count):   {total_events:>12,}  (should equal explode count)")
print(f"  🔁 Rows folded into counters (excess): {excess:>12,}  ({excess/exploded_count*100:.4f}%)")

print(f"\n  ── checkin_count Statistics ──")
print(f"  Max checkin_count:                  {max_count:>12}")
print(f"  Avg checkin_count:                  {avg_count:>12.4f}")
print(f"  Keys with count > 1:               {keys_gt_1:>12,}")

print(f"\n  ── checkin_count Distribution ──")
for row in count_dist:
    pct = row['num_keys'] / silver_count * 100
    print(f"  count={row['checkin_count']}:  {row['num_keys']:>12,} keys  ({pct:.4f}%)")

In [0]:
# ============================================================
# Phase 3: Populate Silver Cleaned Table
# ============================================================
# The target table was created by the 02_Checkin_Silver_Cleaned
# DDL query. We INSERT INTO it (after truncating) to preserve
# the DDL-defined PK/FK constraints and TBLPROPERTIES.

# Truncate first to make this notebook idempotent (re-runnable)
spark.sql("TRUNCATE TABLE yelp_poc_uc.silver_cleaned.yelp_checkin")

# Insert aggregated data
(
    df_silver
    .select("business_id", "checkin_timestamp", "checkin_count")
    .write
    .mode("append")
    .insertInto("yelp_poc_uc.silver_cleaned.yelp_checkin")
)
print("✅ Silver Cleaned table populated: yelp_poc_uc.silver_cleaned.yelp_checkin")

In [0]:
# ============================================================
# ✅ Phase 4: Post-Write Validation
# ============================================================
# Read back from the persisted silver_cleaned table to verify correctness.

silver_table = spark.read.table("yelp_poc_uc.silver_cleaned.yelp_checkin")

sv_count        = silver_table.count()
sv_total_events = int(silver_table.agg(_sum("checkin_count")).collect()[0][0])

# 1. Row count reconciliation (sum of checkin_count should equal exploded count)
print("🔍 Validation Results:")
print(f"   Silver rows (unique keys):  {sv_count:>12,}")
print(f"   Sum of checkin_count:       {sv_total_events:>12,}  (expected: {exploded_count:,})")
assert sv_total_events == exploded_count, "❌ checkin_count sum does not match exploded row count!"
print("   ✅ checkin_count sums reconcile correctly\n")

# 2. Composite key uniqueness
distinct_keys = silver_table.select("business_id", "checkin_timestamp").distinct().count()
assert distinct_keys == sv_count, "❌ Composite key not unique!"
print(f"   Distinct composite keys: {distinct_keys:,} == {sv_count:,} total rows")
print("   ✅ Composite key (business_id, checkin_timestamp) is unique\n")

# 3. Null check on key columns
null_biz   = silver_table.filter(col("business_id").isNull()).count()
null_ts    = silver_table.filter(col("checkin_timestamp").isNull()).count()
null_count = silver_table.filter(col("checkin_count").isNull()).count()
print(f"   Null business_id:        {null_biz}")
print(f"   Null checkin_timestamp:  {null_ts}")
print(f"   Null checkin_count:      {null_count}")
assert null_biz == 0 and null_ts == 0 and null_count == 0, "❌ Nulls found in key columns!"
print("   ✅ No nulls in key columns\n")

# 4. checkin_count validity (all values must be >= 1)
min_count = int(silver_table.agg(_min("checkin_count")).collect()[0][0])
assert min_count >= 1, "❌ Found checkin_count < 1!"
print(f"   Min checkin_count:       {min_count}")
print("   ✅ All checkin_count values >= 1\n")

# 5. Schema verification
print("   Silver Cleaned table schema:")
silver_table.printSchema()